# Week 4 – Member 5: GitHub Actions CI

## Learning goal
Set up Continuous Integration (CI) with GitHub Actions so that Ruff, mypy and pytest run automatically on every pull request. By the end, you should be able to read the workflow and explain why a failing required check blocks a merge.

## 1. What is CI?
**Continuous Integration (CI)** is the practice of automatically checking code changes when developers contribute them. It helps detect problems before changes are merged into the main branch.

A **pull request (PR)** is a proposal to merge changes from one branch into another. A **workflow** is the set of automation instructions GitHub Actions follows. A **runner** is the machine that executes those instructions.

## 2. The complete CI process
1. Create a feature branch.
2. Write or change code and tests.
3. Run checks locally.
4. Commit and push the branch to GitHub.
5. Open a pull request targeting `main`.
6. The `pull_request` event triggers the workflow.
7. GitHub provides a runner (`ubuntu-latest` in our example).
8. The runner checks out the repository.
9. The runner sets up Python and installs dependencies.
10. Ruff, mypy and pytest run.
11. GitHub reports the results on the pull request.
12. Branch protection/rulesets can require those checks to pass before merging.

**Flow:** Edit → local checks → commit → push → PR → workflow trigger → runner setup → install dependencies → Ruff → mypy → pytest → report → merge decision.

## 3. What do the tools check?
- **Ruff:** linting and code-quality issues, such as unused imports and undefined names.
- **mypy:** static type checking; it compares type annotations with how values are used.
- **pytest:** runs tests to check that program behaviour matches expectations.

These tools complement one another. Passing them does not prove that a program is perfect; it means the configured checks passed.


## 4. A tiny Python project

We will use a small function that calculates an average. It is only a practice example; the CI pattern is what matters.

In [ ]:
def calculate_average(scores: list[float]) -> float:
    total = sum(scores)
    return total / len(scores) if scores else 0.0

print(calculate_average([70, 80, 90]))
print(calculate_average([]))

Expected output:
```text
80.0
0.0
```

The annotations `list[float]` and `-> float` describe the intended input and return types. Python does not automatically enforce these annotations at runtime; mypy can check them before execution.

In [ ]:
# A simple test, similar to the contents of test_main.py
def test_calculate_average():
    result = calculate_average([70, 80, 90])
    assert result == 80

test_calculate_average()
print("Test passed")

## 5. pytest: testing behaviour

A test usually has a name beginning with `test_`. The `assert` statement checks whether a condition is true. If it is false, pytest reports a failure.

In the repository, the test belongs in `test_main.py` and imports the function from `main.py`:

```python
from main import calculate_average

def test_calculate_average():
    result = calculate_average([70, 80, 90])
    assert result == 80
```

Run locally:
```powershell
python -m pytest
```

To understand a failure, compare the actual value with the expected value. For example, changing the expected value from `80` to `100` makes this test fail because the function returns `80.0`.


## 6. Ruff: linting

A **linter** checks code for patterns that may indicate mistakes or poor style. Ruff can find issues such as unused imports and undefined names.

Run:
```powershell
python -m ruff check main.py run.py test_main.py
```

Ruff can automatically fix some supported issues with `--fix`, but CI should normally run the check without silently changing files:
```powershell
python -m ruff check main.py run.py test_main.py
```

We deliberately scope this practice command to the files in this exercise so unrelated older files in the repository do not distract from the lesson.


## 7. mypy: static type checking

**Static** means the checker examines code without needing to run the program's normal behaviour. mypy uses annotations to find type mismatches.

Example:
```python
def greet(name: str) -> str:
    return "Hello " + name

greet(42)  # mypy should report that an int was supplied where str is expected
```

Run:
```powershell
python -m mypy main.py
```

Type annotations are useful documentation, but they are not a runtime guarantee by themselves.


## 8. Dependencies and requirements.txt

A dependency is an external package the project needs. Our `requirements.txt` pins the package versions used by this exercise:

```text
pytest==9.2.0
ruff==0.17.5
mypy==2.4.1
```

`==` means an exact version is requested. The GitHub runner is a separate machine from your computer, so it needs its own installation:
```bash
python -m pip install -r requirements.txt
```

The local virtual environment folder `.venv/` should be ignored by Git, not committed.


## 9. The GitHub Actions workflow

Save this as `.github/workflows/ci.yml`:

```yaml
name: Python CI

on:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - name: Get repository code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.14"

      - name: Install dependencies
        run: python -m pip install -r requirements.txt

      - name: Run Ruff
        run: python -m ruff check main.py run.py test_main.py

      - name: Run mypy
        run: python -m mypy main.py

      - name: Run pytest
        run: python -m pytest
```

### Reading the YAML
- `name`: display name of the workflow.
- `on: pull_request`: event that triggers the workflow.
- `jobs`: the jobs GitHub should run.
- `test`: this job's identifier.
- `runs-on`: selects the runner environment. `ubuntu-latest` means a GitHub-hosted Ubuntu runner.
- `steps`: the ordered actions/commands inside the job.
- `uses`: asks GitHub to run a reusable action.
- `actions/checkout@v4`: checks out the repository code onto the runner. `v4` is the action version.
- `actions/setup-python@v5`: sets up Python on the runner.
- `with`: supplies configuration to an action.
- `python-version`: selects the Python version.
- `run`: executes a shell command on the runner.

Steps in one job normally execute in order. If a step fails, later steps in that job are normally skipped unless the workflow is configured otherwise. The order Ruff → mypy → pytest is a sensible choice, not a mandatory universal order.


## 10. Why a failing check blocks a merge

The workflow reports a result, but **branch protection or a repository ruleset** is what makes a check mandatory.

If a required check fails or is still pending, GitHub does not allow the PR to merge while that requirement is unsatisfied. When the required checks pass, the check requirement is satisfied, although other rules (such as approvals) may still apply.

A workflow can run successfully without being a required check. Running checks and enforcing checks are two related but distinct settings.


## 11. Useful local commands

```powershell
python -m pip install -r requirements.txt
python -m ruff check main.py run.py test_main.py
python -m mypy main.py
python -m pytest
git diff --cached --check
```

Before pushing, check the output and fix issues locally. Then commit and push your feature branch and open/update the PR.


## 12. Friday task checklist

- [ ] Add the Python example and its test.
- [ ] Add Ruff, mypy and pytest to `requirements.txt`.
- [ ] Create `.github/workflows/ci.yml`.
- [ ] Run all three checks locally.
- [ ] Commit and push the feature branch.
- [ ] Open a PR targeting `main`.
- [ ] Confirm the workflow runs and passes.
- [ ] Configure the repository ruleset to require the intended check.
- [ ] Be able to explain why a failing required check blocks merging.

## 13. Teach-back questions

1. What is CI, in your own words?
2. What event starts this workflow?
3. What does `runs-on` specify?
4. Why does the runner install dependencies?
5. What does Ruff check?
6. What does mypy check?
7. What does pytest check?
8. What is the difference between a workflow passing and a check being required?
9. Why does a failed required check block merging?
10. Where does the workflow execute: on your laptop or on the runner?

**Key idea:** GitHub Actions runs the commands on the runner selected by `runs-on`. Your laptop is where you develop and can run the same checks before pushing; it is not where the hosted workflow executes.
